In [1]:
# Chap 16. 다양한 댓글 모으기 (다중 쇼츠 연속 수집 아키텍처)
# 테스트 기사 URL : https://www.youtube.com/shorts/UkU_P-jfBtM

# Step 1. 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import pandas as pd
import os
import re

# Step 2. 사용자 설정
print("=" *80)
print(" Chap 16. 쇼츠 댓글 정보 연속 수집하기 (다중 영상)")
print("=" *80)
print("\n")

query_txt = '쇼츠댓글연속수집'
# [아키텍처 변경 1] 다중 영상이므로 특정 영상의 총 댓글 수(page_cnt)에 의존하지 않고, 오직 사용자가 입력한 총 목표(cnt)만 바라봅니다.
cnt = int(input('2. 크롤링 할 "총 댓글 건수"는 몇 건입니까?: '))

f_dir = input("3.파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\):")
if f_dir=='' :
    f_dir='c:\\py_temp\\결과 추출 - 리뷰, 댓글\\'

print(f"\n요청하신 총 {cnt}건의 데이터를 여러 쇼츠를 넘나들며 수집합니다. 잠시만 기다려 주세요~~")

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)
os.makedirs(f_dir+s+'-'+query_txt, exist_ok=True)
os.chdir(f_dir+s+'-'+query_txt)

ff_name = f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name = f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name = f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xls'

# Step 3. 크롬 드라이버 실행
s_time = time.time()
s_drv = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s_drv)

# 첫 번째 쇼츠 진입
driver.get("https://www.youtube.com/shorts/UkU_P-jfBtM")
driver.maximize_window()
time.sleep(5)

# (기존 Step 4의 영상 개수 비교 로직은 다중 영상 수집이므로 과감히 삭제합니다.)

# [아키텍처 변경 2] 최초 1회 댓글창 열기
# 쇼츠는 처음에 댓글창이 닫혀 있으므로 한 번은 열어주어야 합니다. 한 번 열리면 다음 쇼츠로 넘어가도 열린 상태가 유지됩니다.
try:
    driver.find_element(By.XPATH,'//*[@id="comments-button"]/ytd-button-renderer/yt-button-shape/label/button').click()
except:
    pass
time.sleep(3)

# Step 6. 데이터 담을 그릇 및 글로벌 카운터 준비
y_URL2, reviewer2, review_d2, review2, like2 = [], [], [], [], []
global_count = 0 # 전체 수집 건수 (100건, 200건 등 목표치 도달 확인용)
video_num = 1 # 몇 번째 쇼츠인지 확인용

# ==============================================================================
# [핵심 아키텍처: OUTER LOOP (영상 단위 반복)]
# 목표한 global_count가 cnt에 도달할 때까지 계속 다음 쇼츠로 넘기며 반복합니다.
# ==============================================================================
while global_count < cnt:
    current_url = driver.current_url
    print(f"\n{'='*60}")
    print(f"🎬 [{video_num}번째 쇼츠] 수집 시작: {current_url}")
    print(f"{'='*60}")
    
    # [아키텍처 변경 3] 로컬 상태 초기화 (매우 중요)
    # 새로운 영상에 진입했으므로, 이전 영상에서 쓰던 정체 감지 센서를 0으로 리셋합니다.
    stale_retry_count = 0 
    prev_count = -1
    local_count = 0 # 이 영상에서 수집한 댓글 수
    
    # ==========================================================================
    # [핵심 아키텍처: INNER LOOP (단일 영상 내 스크롤 및 추출)]
    # 질문자님의 기존 완벽한 추출 코드가 이 안에서 구동됩니다.
    # ==========================================================================
    while True: # 이 영상의 댓글이 바닥날 때까지 무한 스크롤
        f = open(ff_name, 'a', encoding='UTF-8')
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        # 현재 열려있는 댓글창의 렌더러만 가져옵니다.
        reple_result = soup.find('div', id='contents').find_all('ytd-comment-thread-renderer')
        new_comments = reple_result[local_count:]

        # [조기 종료 안전장치: 현재 영상의 댓글이 바닥났는지 확인]
        if len(new_comments) == 0:
            stale_retry_count += 1
            if stale_retry_count >= 2:
                print(f"✅ [{video_num}번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.")
                f.close()
                break # 이 영상 수집(Inner Loop) 종료 -> 다음 영상으로
        else:
            stale_retry_count = 0

        # [데이터 추출 루프]
        for li in new_comments:
            # 대댓글 제외 로직
            main_comment = li.find('ytd-comment-view-model')
            if not main_comment:
                continue

            local_count += 1
            global_count += 1
            print(f"\n▶ 총 목표 {cnt}건 중 {global_count}번째 데이터 수집 중... (현재 영상 내 {local_count}번째)")
            
            f.write(f"\n총 {cnt} 건 중 {global_count} 번째 리뷰 데이터====\n")
            f.write("1.동영상 URL: " + current_url + "\n")
            y_URL2.append(current_url)
            
            try:
                reviewer = li.find('div', id='header-author').find('a', 'yt-simple-endpoint style-scope ytd-comment-view-model').get_text()
                reviewer = reviewer.replace("\n", "").strip()
            except:
                try:
                    reviewer = li.select_one('#author-text').get_text().replace("\n", "").strip()
                except:
                    reviewer = '작성자 알 수 없음'
            print("2.댓글작성자명:", reviewer)
            f.write("2.댓글작성자명: " + reviewer + "\n")
            reviewer2.append(reviewer)

            try:
                review_d = li.find('div', id='header-author').find('span', id='published-time-text').get_text()
                review_d = review_d.replace("\n", "").strip()
            except:
                review_d = '날짜 알 수 없음'
            print("3.댓글작성일자:", review_d)
            f.write("3.댓글작성일자: " + review_d + "\n")
            review_d2.append(review_d)

            try:
                review = li.find('div', id='content').find('span', 'yt-core-attributed-string').get_text()
                review = review.replace("\n", "").strip()
            except:
                review = '내용 없음'
            print("4.리뷰내용:", review)
            f.write("4.리뷰내용: " + review + "\n")
            review2.append(review)

            try:
                like_node = li.select_one('like-button-view-model span[role="text"]')
                if like_node:
                    like = like_node.get_text(strip=True)
                else:
                    backup_node = li.select_one('#vote-count-middle')
                    like = backup_node.get_text(strip=True) if backup_node else '0'
                if not like:
                    like = '0'
            except:
                like = '0'
            print("5.좋아요횟수:", like)
            f.write("5.좋아요횟수:" + like + "\n")
            like2.append(like)

            # [목표 달성 체크 (강제 종료)]
            if global_count >= cnt:
                f.close()
                break

        f.close()

        # 목표 달성 시 이너 루프도 즉시 탈출
        if global_count >= cnt:
            break

        # 댓글창 내부를 스크롤 다운하기 위해 자바스크립트 실행 (무한 스크롤)
        try:
            driver.execute_script("document.querySelector('#contents').scrollTo(0, document.querySelector('#contents').scrollHeight);")
        except:
            pass
        time.sleep(1.5)

    # ==========================================================================
    # [아우터 루프 후반부: 다음 쇼츠로 화면 넘기기 (Swipe Down)]
    # ==========================================================================
    if global_count >= cnt:
        print(f"\n🎉 목표로 하신 {cnt}건 수집을 완벽하게 달성하여 프로그램을 종료합니다.")
        break # 모든 루프 완전 탈출
        
    print(f"\n🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...")
    try:
        # 키보드 아래 화살표를 눌러 틱톡/쇼츠처럼 다음 영상으로 넘깁니다.
        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.ARROW_DOWN)
    except:
        pass
    
    # 영상이 바뀌고 새로운 댓글이 렌더링될 때까지 충분히 대기합니다.
    time.sleep(4) 
    video_num += 1

# Step 7. 엑셀 및 CSV 저장
news_reple = pd.DataFrame()
news_reple['동영상 URL'] = pd.Series(y_URL2)
news_reple['댓글작성자명'] = pd.Series(reviewer2)
news_reple['댓글 작성일자'] = pd.Series(review_d2)
news_reple['리뷰내용'] = pd.Series(review2)
news_reple['좋아요횟수'] = pd.Series(like2)

news_reple.to_csv(fc_name, encoding="utf-8-sig", index=True)
news_reple.to_excel(fx_name, index=True, engine='openpyxl')

# Step 8. 요약 정보 출력
e_time = time.time()
t_time = e_time - s_time

print("\n" + "="*120)
print(f"1. 총 수집된 댓글 수: {global_count} 건 (방문한 쇼츠 영상 수: {video_num} 개)")
print(f"2. 총 소요시간: {round(t_time,1)} 초")
print(f"3. txt 저장 완료: {ff_name}")
print(f"4. csv 저장 완료: {fc_name}")
print(f"5. xls 저장 완료: {fx_name}")
print("="*120)

driver.quit()

 Chap 16. 쇼츠 댓글 정보 연속 수집하기 (다중 영상)



요청하신 총 100건의 데이터를 여러 쇼츠를 넘나들며 수집합니다. 잠시만 기다려 주세요~~

🎬 [1번째 쇼츠] 수집 시작: https://www.youtube.com/shorts/UkU_P-jfBtM
✅ [1번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.

🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...

🎬 [2번째 쇼츠] 수집 시작: https://www.youtube.com/shorts/w4fNWZ89vpk
✅ [2번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.

🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...

🎬 [3번째 쇼츠] 수집 시작: https://www.youtube.com/shorts/M7Tb1rY6Sdo
✅ [3번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.

🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...

🎬 [4번째 쇼츠] 수집 시작: https://www.youtube.com/shorts/hAJobHa__Tw
✅ [4번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.

🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...

🎬 [5번째 쇼츠] 수집 시작: https://www.youtube.com/shorts/s3Ozo0Jep-o
✅ [5번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.

🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...

🎬 [6번째 쇼츠] 수집 시작: https://www.youtube.com/shorts/uHTKv3pwF_4
✅ [6번째 쇼츠] 더 이상 수집할 댓글이 없습니다. 다음 쇼츠로 이동합니다.

🚀 키보드 '아래 방향키'를 눌러 다음 쇼츠로 슬와이프 합니다...

🎬 [7번째 쇼츠] 수집 시작: https://www.youtube.co

KeyboardInterrupt: 

이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~

[현재 수집 중인 동영상 URL: https://www.youtube.com/shorts/UkU_P-jfBtM]


총 22건 중 1번째 댓글 수집 중입니다 =========
1.동영상 URL: https://www.youtube.com/shorts/UkU_P-jfBtM
2.댓글작성자명: 
 @catch_speed 

3.댓글작성일자: 

            5개월 전
          

4.리뷰내용: 올리브영 알바생이 추천하는 꿀템 5가지

1. 짱구 향수
짱구덕후라면 꼭 샤야된다. 짱구도 귀엽고 파우치 키링까지 주는데 포근한 코튼향이라 내 살냄새같아서 짱조음

2. 콜레올로지 컷팅 젤리 

아이돌 다이어트 필수템인 컷팅 젤리인데 간식 땡길 때 쭉쭉 짜먹으면 좋아요! 체지방 컷팅에 식후혈당까지 다잡아줘서 가방에 하나씩 넣고다니기 좋음! 

3. 도로시와 퍼스널 컬러 컷 누드브라 
너무너무 좋아서 여행갈때도 챙겨갔는데 퍼스널 컬러 컷 누드브라 이름처럼 컬러가 여러개라 
내 피부톤에 찰떡인 컬러로 고르면 티 1도 안남!! 촉감도 부드럽고 얇아서 역대급으로 자연스럽고 라인도이쁨♡

4. 메디큐브 핑크콜라겐 캡슐크림
저 이제 피부과 안갑니다. 분홍색 캡슐을 톡톡 터트리며 발라주면 담날 트러블 다들어가요

5. 레인보우 치즈 베이글칩
치즈맛 제대로나고 바삭바삭한게 진짜 맛있어서 입심심할때 다이어트 과자로 좋아요
5.좋아요횟수: 31


총 22건 중 2번째 댓글 수집 중입니다 =========
1.동영상 URL: https://www.youtube.com/shorts/UkU_P-jfBtM
2.댓글작성자명: 
 @뿡뿡이-d3q 

3.댓글작성일자: 

            1개월 전
          

4.리뷰내용: 저거 캡슐크림 달바꺼도 진짜 좋아요!! 비타민이 여러모로 최고에요.. 갠적으로 스킨케어할 때 화장품도 비타민 성분만 쓰는데 바이오힐보 비타민토너, 비타치올 비타민앰플, 달바 비타크림으



1.요청된 총 22 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 22 건입니다
2.총 소요시간은 275.6 초 입니다 
3.파일 저장 완료: txt 파일명 : c:\py_temp\결과 추출 - 리뷰, 댓글\2026-03-11-17-20-00-쇼츠댓글\2026-03-11-17-20-00-쇼츠댓글.txt 
4.파일 저장 완료: csv 파일명 : c:\py_temp\결과 추출 - 리뷰, 댓글\2026-03-11-17-20-00-쇼츠댓글\2026-03-11-17-20-00-쇼츠댓글.csv 
5.파일 저장 완료: xls 파일명 : c:\py_temp\결과 추출 - 리뷰, 댓글\2026-03-11-17-20-00-쇼츠댓글\2026-03-11-17-20-00-쇼츠댓글.xls 
